In [1]:
import os
import pandas as pd
from pathlib import Path
from PIL import Image

In [2]:
USER_ROOT = Path("/Users/romitbarua/BreastCancerResearch/Ekyaalo-WSI-ROI-Detection/")
DATA_ROOT = USER_ROOT / "data/raw/AnnotatedDataSet"
VALID_EXTENSIONS = ['.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp']

In [3]:

def generate_metadata(root_dir, label_map):

    # Get the classes from the folders
    classes = [d.name for d in DATA_ROOT.iterdir() if d.is_dir()]
    print('Classes:', classes)

    rows = []
    for cls in classes:
        class_dir = DATA_ROOT / cls

        assert cls in label_map, f"Class {cls} not found in label map"
        label = label_map[cls]

        if label == -1:
            continue

        for file in class_dir.iterdir():
            if not file.suffix in VALID_EXTENSIONS:
                print(f"Warning: invalid file extension: {file}")
                continue
            fpath = file.as_posix()
            fname = file.name
            extension = file.suffix

            slide_id = "UNKNOWN"
            patient_id = "UNKNOWN"
            if "Z-" in fname and ".tif" in fname:
                try:
                    slide_id = fname.split("Z-")[1].split(".tif")[0]
                    # Extract patient_id from slide_id
                    patient_id = "-".join(slide_id.split("-")[0:2])
                except Exception:
                    slide_id = fname  # fallback
            else:
                slide_id = fname  # fallback if pattern different

            # Get image dimensions (height, width)
            try:
                with Image.open(fpath) as img:
                    width, height = img.size
            except Exception as e:
                print(f"Warning: Failed to get size for {fpath}: {e}")
                width, height = None, None

            # remove the USER_ROOT from the filepath
            fpath = str(fpath).replace(str(USER_ROOT), "")

            rows.append({
                "filepath": fpath,
                "fname": fname,
                "class_name": cls,
                "label": label,
                "slide_id": slide_id,
                'patient_id': patient_id,
                "width": width,
                "height": height
            })

    df = pd.DataFrame(rows)
    print("Total images:", len(df))
    print(df["label"].value_counts().rename(index={0:"Benign",1:"Susp/Malig"}))
    
    return df


In [4]:
def assign_split(df, test_ratio=0.25, val_ratio=0.25):
    """
    Assigns splits to the dataframe based on the split ratio.
    """
    # Shuffle and split unique slide_ids to avoid leakage between splits
    #slide_ids = df['slide_id'].unique()
    #slide_ids = pd.Series(slide_ids).sample(frac=1, random_state=42)  # shuffle slide_ids

    patient_ids = df['patient_id'].unique()
    patient_ids = pd.Series(patient_ids).sample(frac=1, random_state=42)  # shuffle patient_ids

    key_ids = patient_ids

    n_unique = len(key_ids)
    train_ratio = 1 - test_ratio - val_ratio    
    n_train = int(train_ratio * n_unique)
    n_val = int(val_ratio * n_unique)
    n_test = n_unique - n_train - n_val

    #train_slide_ids = set(slide_ids.iloc[:n_train])
    #test_slide_ids = set(slide_ids.iloc[n_train:n_train+n_test])
    #val_slide_ids = set(slide_ids.iloc[n_train+n_test:])

    train_slide_ids = set(patient_ids.iloc[:n_train])
    test_slide_ids = set(patient_ids.iloc[n_train:n_train+n_test])
    val_slide_ids = set(patient_ids.iloc[n_train+n_test:])

    df['split'] = df['patient_id'].apply(lambda x: 'train' if x in train_slide_ids else 'test' if x in test_slide_ids else 'val')

    return df
    

In [5]:
label_codes_one = {'Malignant': 1, 'Benign': 0, 'Suspicious': 1}
label_codes_two = {'Malignant': 1, 'Benign': 0, 'Suspicious': -1}

df = generate_metadata(
    DATA_ROOT,
    label_codes_two
)
df = assign_split(df)
df.to_csv('/Users/romitbarua/BreastCancerResearch/Ekyaalo-WSI-ROI-Detection/data/metadata/metadata.csv')

Classes: ['Malignant', 'Benign', 'Suspicious']
Total images: 2970
label
Susp/Malig    1734
Benign        1236
Name: count, dtype: int64


In [16]:
df = pd.read_csv('/Users/romitbarua/BreastCancerResearch/Ekyaalo-WSI-ROI-Detection/data/metadata/metadata.csv')

In [23]:
# Generate the Patient ID from slide_id in a vectorized and robust manner
df['patient_id'] = df['slide_id'].apply(lambda x: '-'.join(str(x).split('-')[:2]) if pd.notnull(x) else None)
df = assign_split(df)
df.to_csv('/Users/romitbarua/BreastCancerResearch/Ekyaalo-WSI-ROI-Detection/data/metadata/metadata.csv')


In [24]:
df.head()

,filepath,fname,class_name,label,slide_id,width,height,patient_id,split
0,/data/raw/AnnotatedDataSet/Benign/2025-04-16T2...,2025-04-16T21-36-24-411Z-C1432-23-1.tif_lux_14...,Benign,0,C1432-23-1,400,400,C1432-23,train
1,/data/raw/AnnotatedDataSet/Benign/2025-04-16T2...,2025-04-16T21-36-24-455Z-C1131-23-2.tif_lux_29...,Benign,0,C1131-23-2,400,400,C1131-23,test
2,/data/raw/AnnotatedDataSet/Benign/2025-04-16T2...,2025-04-16T21-36-24-922Z-C1265-23-1.tif_lux_15...,Benign,0,C1265-23-1,400,400,C1265-23,test
3,/data/raw/AnnotatedDataSet/Benign/2025-04-16T2...,2025-04-16T21-36-24-972Z-C1103-23-1.tif_lux_14...,Benign,0,C1103-23-1,400,400,C1103-23,test
4,/data/raw/AnnotatedDataSet/Benign/2025-04-16T2...,2025-04-16T21-36-25-010Z-C1131-23-2.tif_lux_21...,Benign,0,C1131-23-2,400,400,C1131-23,test


In [25]:
df['slide_id'].unique()

array(['C1432-23-1', 'C1131-23-2', 'C1265-23-1', 'C1103-23-1',
       'C775-22-1', 'c1106-23-1', 'C016-23-1', 'C016-23-3', 'C857-22-1',
       'C1104-23-1', 'FNAC214-21-1', 'C1105-23-1', 'C1336-22-1',
       'C1102-23-1', 'C1336-22-2', 'C1357-22-1', 'C1405-22-1',
       'C1405-22-2', 'C1405-22-3', 'C1432-23-2', 'C1453-23-1',
       'C1453-23-2', 'C1496-22-2', 'C1907-22-1', 'C1907-22-2',
       'C1908-22-1', 'C1930-23-1', 'C201-23-1', 'C2192-23-2', 'C410-22-1',
       'C418-22-1', 'C443-22-1', 'C443-22-2', 'C445-22-1', 'C445-22-2',
       'C501-22-1', 'C501-22-2', 'C51-23-2', 'C771-22-1', 'C771-22-2',
       'C775-22-2', 'C782-22-1', 'C804-22-1', 'C804-22-2', 'C825-22-1',
       'C825-22-2', 'C825-22-3', 'C857-22-2', 'C864-22-1', 'C875-22-1',
       'C875-22-2', 'FNAC145-21-1', 'FNAC146-21-2', 'FNAC209-21-1',
       'FNAC214-21-2', 'FNAC215-21-1', 'FNAC215-21-2', 'FNAC87-21-1',
       'C2007-23-1', 'C016-23-2', 'C1102-23-2', 'C1105-23-2',
       'C1106-23-1', 'C1106-23-2', 'C418-22-2', 

In [26]:
df.groupby(['patient_id', 'split']).count()

,,filepath,fname,class_name,label,slide_id,width,height
patient_id,split,,,,,,,
C016-23,train,100,100,100,100,100,100,100
C1102-23,train,76,76,76,76,76,76,76
C1103-23,test,58,58,58,58,58,58,58
C1104-23,train,12,12,12,12,12,12,12
C1105-23,val,17,17,17,17,17,17,17
C1106-23,train,52,52,52,52,52,52,52
C1131-23,test,79,79,79,79,79,79,79
C1265-23,test,49,49,49,49,49,49,49
C1336-22,test,150,150,150,150,150,150,150


In [27]:
df.groupby(['split']).count()

,filepath,fname,class_name,label,slide_id,width,height,patient_id
split,,,,,,,,
test,687,687,687,687,687,687,687,687
train,1244,1244,1244,1244,1244,1244,1244,1244
val,1039,1039,1039,1039,1039,1039,1039,1039
